# StereoQueerEval 2027 — Task B: Dual-Stream Asymmetric Cross-Context Architecture

Notebook này huấn luyện mô hình **Task B (Hate Speech: `no`, `yes_implicit`, `yes_explicit`)** theo kiến trúc **Dual-Stream Asymmetric Cross-Context Network**:

```
                ┌───────────────────┴───────────────────┐
                ▼                                       ▼
     STREAM 1: TITLE + COMMENT                 STREAM 2: DESCRIPTION
     [CLS] Title [SEP] Comment [SEP]           [CLS] Description [SEP]
                │                                       │
             mmBERT                                  mmBERT
          (Shared Wts)                            (Shared Wts)
                │                                       │
                ▼                                       ▼
              H_TC                                     H_D
          ┌─────┴─────┐                                 │
          ▼           ▼                                 │
       H_title     H_comment                            │
          │           │                                 │
          │           ▼                                 ▼
          │     Multi-Head Cross-Attn (Q = H_comment, K,V = H_D)
          │                   │
          │                   ▼
          │               Context C
          │                   │
          │                   ▼
          │     Element-wise Gated Residual Fusion: g = σ(W[H; C] - 1.5)
          │     H'_comm = RMSNorm(H_comment + g ⊙ C)
          │                   │
          ▼                   ▼
     Masked Pool(H_title)  Masked Pool(H'_comm)
          └───────────┬───────┘
                      │ Concat [H_title; H'_comm] ∈ [B, 1536]
                      ▼
          Multi-Sample Dropout Classifier (5 heads, p=0.1..0.5)
                      │
                      ▼
           Logits ∈ [B, 3] (0: No, 1: Implicit, 2: Explicit)
```

---
### 🌟 Lợi thế vượt trội của kiến trúc này:
1. **Không bị ô nhiễm bởi Description rác**: Description chứa link/promo được cô lập ở luồng riêng, không làm loãng Token Budget của Title & Comment.
2. **Phụ thuộc bất đối xứng (Asymmetric Dependency)**: Chỉ có Comment truy vấn Description để lấy bối cảnh làm rõ nghĩa ẩn ý.
3. **Cổng Gate $g \in (0, 1)^d$ thông minh**: Tự động bật khi gặp Implicit Hate cần ngữ cảnh, và tự động đóng ($g \to 0$) khi gặp Explicit Hate rõ ràng hoặc Description toàn rác.

## 1. Clone hoặc Cập nhật Mã Nguồn từ GitHub

In [ ]:
import os
import sys

REPO_URL = "https://github.com/AmnO-O/lgbtia-.git"
REPO_DIR = "stereoqueer-pipeline"

if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL}...")
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}
else:
    print(f"Repository directory '{REPO_DIR}' already exists.")
    if os.path.isdir(REPO_DIR):
        %cd {REPO_DIR}
        !git pull

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print("Current Working Directory:", os.getcwd())
!ls -la

## 2. Cài đặt Thư viện & Chạy Smoke Test

In [ ]:
!pip install -q torch transformers datasets accelerate scikit-learn pandas numpy matplotlib seaborn

import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
    print("Memory Allocated:", round(torch.cuda.memory_allocated(0)/1024**3, 2), "GB")

# Chạy smoke test toàn bộ kiến trúc Dual-Stream Cross-Context
!python3 tests/test_task_b_cross_context.py

## 3. Cấu hình Hyperparameters cho Dual-Stream Training

In [ ]:
from pipeline.config import PipelineConfig

config = PipelineConfig()

# Backbone
config.embed_source = 'mmbert'
config.mmbert_model_name = 'answerdotai/ModernBERT-base' # hoặc 'xlm-roberta-base'
config.d_model = 768

# Dual-Stream Sequence Lengths
TC_MAX_LEN = 128   # Title (max 48 tok) + Comment (max 80 tok)
DESC_MAX_LEN = 128 # Description context

# Training Hyperparameters
config.batch_size = 32
config.two_phase = True
config.freeze_phase_epochs = 3     # Phase 1: Freeze backbone, train cross-attn + gate
config.unfreeze_phase_epochs = 15   # Phase 2: Unfreeze top layers
config.unfreeze_layers = 4         # Unfreeze top 4 encoder layers

config.learning_rate = 2.5e-4      # Phase 1 Head LR
config.unfreeze_lr = 1.5e-5        # Phase 2 Backbone LR
config.head_unfreeze_lr = 1.0e-4   # Phase 2 Head LR
config.weight_decay = 0.01
config.warmup_ratio = 0.10

# Regularization & Losses
config.use_amp = True
config.use_fgm = True
config.fgm_epsilon = 0.50
config.loss_type = 'focal'
config.output_dir = 'checkpoints_task_b_cross'

print("Cấu hình đã sẵn sàng!")

## 4. Tải Dữ liệu & Khởi tạo Dual-Stream DataLoaders

Sử dụng GroupShuffleSplit (theo video title) đảm bảo **0% data leakage** giữa tập Train và tập Val.

In [ ]:
from transformers import AutoTokenizer
from pipeline.data import DataPipeline
from pipeline.task_b_cross_data import create_task_b_dual_stream_loaders
import numpy as np
import torch

# 1. Tokenizer
tokenizer = AutoTokenizer.from_pretrained(config.mmbert_model_name)

# 2. Load & Split Data
data_pipe = DataPipeline(config)
df_all = data_pipe.load_data()
df_train, df_val = data_pipe.split_data(test_size=0.15, random_state=42)

# 3. Compute Class Weights (Tăng trọng số cho lớp 1: Implicit Hate)
train_labels = df_train['hs_y'].values
class_counts = np.bincount(train_labels, minlength=3)
total_samples = len(train_labels)
class_weights = total_samples / (3.0 * class_counts + 1e-6)
# Tăng ưu tiên lớp implicit
class_weights[1] *= 1.25
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

print(f"Phân bố nhãn Train: No={class_counts[0]}, Implicit={class_counts[1]}, Explicit={class_counts[2]}")
print(f"Class Weights: {class_weights_tensor.numpy()}")

# 4. DataLoaders
train_loader, val_loader = create_task_b_dual_stream_loaders(
    df_train=df_train,
    df_val=df_val,
    tokenizer=tokenizer,
    batch_size=config.batch_size,
    tc_max_len=TC_MAX_LEN,
    desc_max_len=DESC_MAX_LEN
)

print(f"Train Batches: {len(train_loader)} | Val Batches: {len(val_loader)}")

## 5. Khởi tạo Model Dual-Stream Cross-Context & Trainer

In [ ]:
from transformers import AutoModel
from pipeline.models.task_b_cross_context import TaskBCrossContextAttentionModel
from pipeline.task_b_cross_trainer import TaskBCrossContextTrainer

# 1. Load Pretrained Backbone
print(f"Loading backbone: {config.mmbert_model_name}...")
mmbert_backbone = AutoModel.from_pretrained(config.mmbert_model_name)

# 2. Build Dual-Stream Model
model = TaskBCrossContextAttentionModel(
    mmbert_model=mmbert_backbone,
    d_model=config.d_model,
    num_classes=3,
    num_heads=8,
    dropout=0.20,
    gate_bias_init=-1.50,
    msd_dropout_rates=[0.1, 0.2, 0.3, 0.4, 0.5]
)

# 3. Trainer
trainer = TaskBCrossContextTrainer(
    model=model,
    config=config,
    train_loader=train_loader,
    val_loader=val_loader,
    class_weights=class_weights_tensor
)

print("Model & Trainer đã sẵn sàng!")

## 6. Thực thi Huấn luyện Two-Phase Training

In [ ]:
results = trainer.train()

print("\n==================================================")
print(f"🏆 BEST TASK B VAL MACRO-F1: {results['best_macro_f1']:.4f}")
print(f"📁 Best Checkpoint: {results['checkpoint_path']}")
print("==================================================")

## 7. Trực quan hóa Lịch sử Huấn luyện, Confusion Matrix & Gate Activation

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

hist_df = pd.DataFrame(results['history'])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Loss & Macro-F1
axes[0].plot(hist_df['epoch'], hist_df['train_loss'], 'o-', label='Train Loss', color='gray')
axes[0].plot(hist_df['epoch'], hist_df['val_loss'], 'o-', label='Val Loss', color='crimson')
axes[0].set_title('Loss Curve')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Per-Class F1 Progression
axes[1].plot(hist_df['epoch'], hist_df['hs_macro_f1'], 's-', label='Macro-F1', color='black', linewidth=2)
axes[1].plot(hist_df['epoch'], hist_df['hs_f1_no'], '^-', label='No-Hate F1', color='green')
axes[1].plot(hist_df['epoch'], hist_df['hs_f1_implicit'], 'd-', label='Implicit F1', color='orange')
axes[1].plot(hist_df['epoch'], hist_df['hs_f1_explicit'], 'v-', label='Explicit F1', color='purple')
axes[1].set_title('Validation F1 Progression')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('F1 Score')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Plot 3: Gate Activation Progression (Mức độ sử dụng Description)
if 'gate_mean' in hist_df.columns:
    axes[2].plot(hist_df['epoch'], hist_df['gate_mean'], 'o-', label='Gate Activation g', color='royalblue', linewidth=2)
    axes[2].set_title('Description Gate g Activation')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('Average g ∈ (0, 1)')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Confusion Matrix
y_true = results['y_true']
y_pred = results['y_pred']
cm = confusion_matrix(y_true, y_pred)
target_names = ['No Hate', 'Implicit Hate', 'Explicit Hate']

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
plt.title('Task B Final Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Ground Truth')
plt.show()

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=target_names, digits=4))